# RAG Lab: Relevance Scoring & Rerankers
##  Implementation 

This notebook implements a complete RAG system with reranking.

**What we'll do:**
1. Load and prepare documents
2. Generate embeddings
3. Store in vector database (Pinecone)
4. Search documents
5. Rerank with Cohere
6. Compare results
7. Evaluate performance

## STEP 0: Install Required Libraries

**IMPORTANT:** Run this cell FIRST, before anything else!

If you get an error about missing modules, this cell will fix it.

In [1]:
import subprocess
import sys

# List of packages to install
packages = [
    'python-dotenv',
    'openai',
    'cohere',
    'pinecone',
    'pdfplumber',
    'pandas'
]

print("📦 Installing required packages...\n")

print("  Cleaning up old Pinecone packages...")
for package_name in ("pinecone-client", "pinecone"):
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", package_name, "-y", "-q"])
    except subprocess.CalledProcessError:
        pass

for package in packages:
    print(f"  ⏳ Installing {package}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", package, "-q"])
    print(f"  ✅ {package} installed\n")

print("="*60)
print("✅ ALL PACKAGES INSTALLED SUCCESSFULLY!")
print("="*60)
print("\nRestart the kernel before running the Pinecone import cell.")

📦 Installing required packages...

  Cleaning up old Pinecone packages...
  ⏳ Installing python-dotenv...
  ✅ python-dotenv installed

  ⏳ Installing openai...
  ✅ openai installed

  ⏳ Installing cohere...
  ✅ cohere installed

  ⏳ Installing pinecone...
  ✅ pinecone installed

  ⏳ Installing pdfplumber...
  ✅ pdfplumber installed

  ⏳ Installing pandas...
  ✅ pandas installed

✅ ALL PACKAGES INSTALLED SUCCESSFULLY!

Restart the kernel before running the Pinecone import cell.


## STEP 1: Setup & Load API Keys

In [2]:
import os
from dotenv import load_dotenv
import json
from typing import List, Dict, Any
import pandas as pd

# Load environment variables from .env file
load_dotenv()

# Get API keys
openai_api_key = os.getenv("OPENAI_API_KEY")
cohere_api_key = os.getenv("COHERE_API_KEY")
pinecone_api_key = os.getenv("PINECONE_API_KEY")

# Verify keys are loaded
print("✅ API Keys Status:")
print(f"  OpenAI: {'✓ Loaded' if openai_api_key else '✗ Missing'}")
print(f"  Cohere: {'✓ Loaded' if cohere_api_key else '✗ Missing'}")
print(f"  Pinecone: {'✓ Loaded' if pinecone_api_key else '✗ Missing'}")

# Check if .env file exists
if not os.path.exists('.env'):
    print("\n⚠️  WARNING: .env file not found in current directory")
    print("   Make sure your .env file is in the same folder as this notebook!")
else:
    print("\n✅ .env file found!")

✅ API Keys Status:
  OpenAI: ✓ Loaded
  Cohere: ✓ Loaded
  Pinecone: ✓ Loaded

✅ .env file found!


## STEP 2: Load and Prepare Documents

In [3]:
pdf_file = "eu_ai_act.pdf"  # The real assignment document

print(f"📄 Loading EU AI Act Document")
print(f"   File: {pdf_file}")
print()

# Check if file exists
if os.path.exists(pdf_file):
    print(f"✅ Found: {pdf_file}")
else:
    print(f"❌ File not found: {pdf_file}")
    print()
    print("📋 PDF files in current directory:")
    pdf_files = [f for f in os.listdir('.') if f.endswith('.pdf')]
    if pdf_files:
        for file in pdf_files:
            print(f"  - {file}")
    else:
        print("  (No PDF files found)")
    print()
    print("👉 Please ensure eu_ai_act.pdf is in the same directory as this notebook!")

📄 Loading EU AI Act Document
   File: eu_ai_act.pdf

✅ Found: eu_ai_act.pdf


In [4]:
import pdfplumber
from itertools import groupby

# Font size -> hierarchy level
SIZE_TO_LEVEL = {
    12.0: "rule",     # Top level
    10.0: "section",  # Mid level
    9.0: None,         # Article level
}
SIZE_TOLERANCE = 0.5

def is_bold(fontname: str | None) -> bool:
    fontname = fontname or ""
    return "Bold" in fontname or fontname.endswith("-BoldMT")

def _level_from_size(size: float):
    closest = min(SIZE_TO_LEVEL, key=lambda candidate: abs(candidate - size))
    if abs(closest - size) <= SIZE_TOLERANCE:
        return SIZE_TO_LEVEL[closest]
    return None

def extract_structure(pdf_path: str):
    """Extract PDF structure with metadata."""
    sections = []
    current: dict | None = None
    found_text = False

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            words = page.extract_words(extra_attrs=["fontname", "size"]) or []

            # Some PDFs expose a text layer only through extract_text(), so use
            # that as a fallback before giving up on the page.
            if not words:
                page_text = (page.extract_text() or "").strip()
                if not page_text:
                    continue
                words = [
                    {"text": line.strip(), "fontname": "", "size": 0.0}
                    for line in page_text.splitlines()
                    if line.strip()
                ]

            found_text = True
            for (size, bold), group in groupby(
                words, key=lambda w: (round(float(w.get("size", 0.0)), 1), is_bold(w.get("fontname")))
            ):
                text = " ".join(w["text"] for w in group).strip()
                if not text:
                    continue

                level = _level_from_size(size)

                if level == "rule":
                    if current:
                        sections.append(current)
                    current = {"level": "rule", "heading": text, "body": [], "page": page_num + 1}

                elif level == "section":
                    if current:
                        sections.append(current)
                    current = {"level": "section", "heading": text, "body": [], "page": page_num + 1}

                elif level is None and bold:
                    if current:
                        sections.append(current)
                    current = {"level": "article", "heading": text, "body": [], "page": page_num + 1}

                else:
                    if current is None:
                        current = {"level": "body", "heading": "", "body": [], "page": page_num + 1}
                    current["body"].append(text)

    if current:
        sections.append(current)

    if not found_text:
        raise RuntimeError(
            "No extractable text was found in this PDF. The file likely needs OCR or a different source document."
        )

    return sections

# Extract structure (UPDATE THE FILENAME ABOVE FIRST!)
print(f"Loading PDF: {pdf_file}...")
try:
    chunks = extract_structure(pdf_file)
    if not chunks:
        raise RuntimeError("The PDF loaded successfully, but no chunks could be extracted.")
    print(f"✅ Extracted {len(chunks)} chunks from PDF")
except FileNotFoundError:
    print(f"❌ File not found: {pdf_file}")
    print("   Please update the filename at the top of this section!")
    chunks = []
except Exception as e:
    print(f"❌ Error: {e}")
    chunks = []

Loading PDF: eu_ai_act.pdf...
✅ Extracted 807 chunks from PDF


In [5]:
# Convert PDF chunks to documents with metadata

if chunks:
    documents = []
    
    for chunk in chunks:
        doc = {
            "text": chunk["heading"],           # Main content
            "source_type": "legal_document",     # Type: legal or podcast
            "hierarchy_level": chunk["level"],  # rule, section, article, body
            "page": chunk["page"],              # Page number
            "body": " ".join(chunk.get("body", [])) if chunk.get("body") else ""  # Additional context
        }
        documents.append(doc)
    
    print(f"✅ Created {len(documents)} documents with metadata")
    print(f"\nSample document:")
    print(json.dumps(documents[0], indent=2))
    
    # Statistics
    print(f"\n📊 Document Statistics:")
    print(f"  Total documents: {len(documents)}")
    print(f"  Hierarchy levels: {set(d['hierarchy_level'] for d in documents)}")
else:
    print("❌ No documents loaded. Please fix the PDF filename above and run again.")
    documents = []

✅ Created 807 documents with metadata

Sample document:
{
  "text": "REGULATION (EU) 2024/1689 OF THE EUROPEAN PARLIAMENT AND OF THE COUNCIL of 13 June 2024 laying down harmonised rules on artificial intelligence and amending Regulations (EC) No 300/2008, (EU) No 167/2013, (EU) No 168/2013, (EU) 2018/858, (EU) 2018/1139 and (EU) 2019/2144 and Directives 2014/90/EU, (EU) 2016/797 and (EU) 2020/1828 (Artificial Intelligence Act)",
  "source_type": "legal_document",
  "hierarchy_level": "section",
  "page": 1,
  "body": ""
}

📊 Document Statistics:
  Total documents: 807
  Hierarchy levels: {'section', 'article'}


## STEP 3: Generate Embeddings with OpenAI

In [6]:
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=openai_api_key)

def get_embeddings(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    """
    Convert text to embeddings (numerical vectors).
    
    Args:
        texts: List of text strings
        model: Which embedding model to use (small is faster/cheaper)
    
    Returns:
        List of embeddings (each is a list of floats)
    """
    response = client.embeddings.create(
        input=texts,
        model=model
    )
    return [item.embedding for item in response.data]

if documents:
    # Generate embeddings for all documents
    print(f"🔄 Generating embeddings for {len(documents)} documents...")
    print("   (This may take a moment)\n")
    
    texts = [doc["text"] for doc in documents]
    embeddings = get_embeddings(texts)
    
    # Add embeddings to documents
    for doc, embedding in zip(documents, embeddings):
        doc["embedding"] = embedding
    
    print(f"✅ Created {len(embeddings)} embeddings")
    print(f"\n📏 Embedding dimensions: {len(embeddings[0])}")
    print(f"   (Each text is now represented as {len(embeddings[0])} numbers)")
else:
    print("⚠️  No documents to embed. Load documents first in STEP 2.")

🔄 Generating embeddings for 807 documents...
   (This may take a moment)

✅ Created 807 embeddings

📏 Embedding dimensions: 1536
   (Each text is now represented as 1536 numbers)


## STEP 4: Store Embeddings in Pinecone Vector Database

In [7]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=pinecone_api_key)
index_name = "legal-rag-index"

try:
    pc.create_index(
        name=index_name,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    print(f"✅ Index '{index_name}' created successfully!")
except Exception as e:
    if "already exists" in str(e).lower() or "ALREADY_EXISTS" in str(e):
        print(f"ℹ️ Index '{index_name}' already exists. Skipping creation.")
    else:
        raise  # Re-raise if it's a different error

# 🔗 Connect to the index for your RAG pipeline
index = pc.Index(index_name)
print("✅ Index connected and ready for upsert/query operations!")

ℹ️ Index 'legal-rag-index' already exists. Skipping creation.
✅ Index connected and ready for upsert/query operations!


In [8]:
if documents and index:
    # Prepare vectors for Pinecone (format must be exact)
    print(f"📦 Preparing {len(documents)} vectors for upload...\n")
    
    vectors_to_upsert = []
    
    for i, doc in enumerate(documents):
        if "embedding" in doc:  # Make sure embedding exists
            vector = {
                "id": f"doc-{i:04d}",  # Unique ID
                "values": doc["embedding"],  # The embedding vector
                "metadata": {  # Information about the document
                    "text": doc["text"][:1000],  # First 1000 chars
                    "source_type": doc["source_type"],
                    "hierarchy_level": doc["hierarchy_level"],
                    "page": int(doc["page"])
                }
            }
            vectors_to_upsert.append(vector)
    
    print(f"Sample vector being uploaded:")
    print(f"  ID: {vectors_to_upsert[0]['id']}")
    print(f"  Vector size: {len(vectors_to_upsert[0]['values'])}")
    print(f"  Metadata keys: {list(vectors_to_upsert[0]['metadata'].keys())}")

## Upload to Pinecone
if index is None:
    print("⚠️  Pinecone index is not connected. Complete STEP 4 first.")
elif not vectors_to_upsert:
    print("⚠️  No vectors to upload. Complete STEP 2 first.")
else:
    print(f"\n⬆️  Uploading {len(vectors_to_upsert)} vectors to Pinecone...")
    try:
        index.upsert(vectors=vectors_to_upsert, namespace="default")
        print(f"✅ Successfully uploaded {len(vectors_to_upsert)} vectors!")
    except Exception as e:
        print(f"❌ Upload error: {e}")

    

📦 Preparing 807 vectors for upload...

Sample vector being uploaded:
  ID: doc-0000
  Vector size: 1536
  Metadata keys: ['text', 'source_type', 'hierarchy_level', 'page']

⬆️  Uploading 807 vectors to Pinecone...
❌ Upload error: [400] Error, decoded message length too large: found 6643724 bytes, the limit is: 4194304 bytes


## STEP 5: Implement Basic Search (BASELINE)

In [9]:
def basic_search(
    query: str,
    top_k: int = 5,
    source_type_filter: str = None
) -> List[Dict[str, Any]]:
    """
    Search documents using embedding similarity.
    
    This is the BASELINE - no reranking yet.
    
    Args:
        query: What to search for
        top_k: How many results to return
        source_type_filter: Filter by source type (optional)
    
    Returns:
        List of search results with scores
    """
    
    # Step 1: Convert query to embedding
    query_embedding = get_embeddings([query])[0]
    
    # Step 2: Search in Pinecone
    filter_dict = None
    if source_type_filter:
        filter_dict = {"source_type": {"$eq": source_type_filter}}
    
    search_results = index.query(
        vector=query_embedding,
        top_k=top_k,
        filter=filter_dict,
        include_metadata=True,
        namespace="default"
    )
    
    # Step 3: Format results nicely
    results = []
    for match in search_results.get("matches", []):
        results.append({
            "id": match["id"],
            "similarity_score": round(match["score"], 4),  # 0-1, higher = better
            "text": match["metadata"]["text"],
            "source_type": match["metadata"]["source_type"],
            "hierarchy_level": match["metadata"]["hierarchy_level"],
            "page": match["metadata"]["page"]
        })
    
    return results

# Test basic search
if index and documents:
    print("🔍 Testing Basic Search (BASELINE)\n")
    print("="*70)
    
    test_query = "What are the penalties for AI violations?"
    print(f"Query: {test_query}\n")
    
    baseline_results = basic_search(test_query, top_k=5)
    
    for i, result in enumerate(baseline_results, 1):
        print(f"{i}. [Similarity: {result['similarity_score']:.4f}]")
        print(f"   {result['text'][:80]}...")
        print(f"   Page {result['page']} | {result['hierarchy_level'].upper()}")
        print()
else:
    print("⚠️  Complete previous steps first (documents + Pinecone upload)")

🔍 Testing Basic Search (BASELINE)

Query: What are the penalties for AI violations?

1. [Similarity: 0.6313]
   Prohibited AI practices...
   Page 51 | ARTICLE

2. [Similarity: 0.5276]
   Obligations of pro viders of high-r isk AI systems...
   Page 62 | ARTICLE

3. [Similarity: 0.5216]
   Procedure at national lev el f or dealing with AI systems presenting a r isk...
   Page 106 | ARTICLE

4. [Similarity: 0.5074]
   Classification of AI sy stems as high-r isk...
   Page 53 | ARTICLE

5. [Similarity: 0.5000]
   Compliant AI systems which present a r isk...
   Page 108 | ARTICLE



## STEP 6: Implement Reranking with Cohere

In [11]:
import cohere

# Initialize Cohere
co = cohere.ClientV2(api_key=cohere_api_key)

def rerank_results(
    query: str,
    results: List[Dict[str, Any]],
    top_k: int = 3
) -> List[Dict[str, Any]]:
    """
    Use Cohere to rerank search results.
    
    Cohere evaluates how relevant each result is to the query
    and returns them in better order.
    
    Args:
        query: The original search query
        results: Results from basic search
        top_k: How many reranked results to return
    
    Returns:
        Reranked results (best first)
    """
    
    if not results:
        return []
    
    # Extract text from results for reranking
    documents = [result["text"] for result in results]
    
    # Send to Cohere for reranking
    try:
        rerank_response = co.rerank(
            model="rerank-english-v3.0",
            query=query,
            documents=documents,
            top_n=top_k
        )
    except Exception as e:
        print(f"❌ Reranking error: {e}")
        return results[:top_k]
    
    # Build reranked results
    reranked_results = []
    for rank_result in rerank_response.results:
        original_result = results[rank_result.index]
        reranked_results.append({
            **original_result,
            "rerank_score": round(rank_result.relevance_score, 4),
            "original_rank": rank_result.index + 1,  # What position it was before
        })
    
    return reranked_results

print("✅ Cohere reranker ready!")

✅ Cohere reranker ready!


## STEP 7: Compare Results - WITH vs WITHOUT Reranking

In [12]:
# Test reranking on the same query

if index and documents:
    print("="*70)
    print("COMPARISON: Reranking Impact")
    print("="*70)
    
    test_query = "What are the penalties for AI violations?"
    print(f"\nQuery: {test_query}\n")
    
    # Without reranking
    print("\n❌ WITHOUT Reranking (Basic Similarity Search)")
    print("-" * 70)
    baseline = basic_search(test_query, top_k=5)
    for i, result in enumerate(baseline, 1):
        print(f"{i}. [Score: {result['similarity_score']:.4f}] {result['text'][:70]}...")
    
    # With reranking
    print("\n✅ WITH Reranking (Cohere)")
    print("-" * 70)
    reranked = rerank_results(test_query, baseline, top_k=3)
    for i, result in enumerate(reranked, 1):
        print(f"{i}. [Cohere: {result['rerank_score']:.4f}] {result['text'][:70]}...")
        if result['original_rank'] != i:
            print(f"   ↑ Moved from position {result['original_rank']}")
        print()
else:
    print("⚠️  Complete previous steps first")

COMPARISON: Reranking Impact

Query: What are the penalties for AI violations?


❌ WITHOUT Reranking (Basic Similarity Search)
----------------------------------------------------------------------
1. [Score: 0.6313] Prohibited AI practices...
2. [Score: 0.5276] Obligations of pro viders of high-r isk AI systems...
3. [Score: 0.5216] Procedure at national lev el f or dealing with AI systems presenting a...
4. [Score: 0.5074] Classification of AI sy stems as high-r isk...
5. [Score: 0.5000] Compliant AI systems which present a r isk...

✅ WITH Reranking (Cohere)
----------------------------------------------------------------------
1. [Cohere: 0.0367] Prohibited AI practices...

2. [Cohere: 0.0004] Compliant AI systems which present a r isk...
   ↑ Moved from position 5

3. [Cohere: 0.0004] Classification of AI sy stems as high-r isk...
   ↑ Moved from position 4



## STEP 8: Complete RAG Pipeline

In [13]:
def rag_pipeline(
    query: str,
    use_reranking: bool = True,
    top_k_search: int = 5,
    top_k_final: int = 3,
    source_type_filter: str = None
) -> Dict[str, Any]:
    """
    Complete RAG pipeline:
    1. Search for relevant documents
    2. Optionally rerank with Cohere
    3. Return top results
    
    Args:
        query: Search query
        use_reranking: Whether to use Cohere reranking
        top_k_search: How many results to get from search
        top_k_final: How many final results to return
        source_type_filter: Filter by source type
    
    Returns:
        Dictionary with results and metadata
    """
    
    # Step 1: Basic search
    search_results = basic_search(
        query,
        top_k=top_k_search,
        source_type_filter=source_type_filter
    )
    
    # Step 2: Optional reranking
    if use_reranking and search_results:
        final_results = rerank_results(query, search_results, top_k=top_k_final)
        method = "WITH Reranking (Cohere)"
    else:
        final_results = search_results[:top_k_final]
        method = "WITHOUT Reranking (Similarity Only)"
    
    return {
        "query": query,
        "method": method,
        "results": final_results,
        "count": len(final_results)
    }

print("✅ RAG Pipeline ready!")

✅ RAG Pipeline ready!


## STEP 9: Evaluation with Multiple Queries

In [14]:
if index and documents:
    # Test queries relevant to EU AI Act
    test_queries = [
        "What are the penalties for AI violations?",
        "What is high-risk AI?",
        "Transparency requirements for AI systems",
        "Prohibited AI practices",
        "How should AI be audited?"
    ]
    
    print("\n" + "="*70)
    print("RAG SYSTEM EVALUATION: Testing Multiple Queries")
    print("="*70)
    
    evaluation_results = []
    
    for query in test_queries:
        print(f"\n📌 Query: {query}")
        print("-" * 70)
        
        # Without reranking
        without = rag_pipeline(
            query,
            use_reranking=False,
            top_k_search=5,
            top_k_final=3
        )
        
        # With reranking
        with_rerank = rag_pipeline(
            query,
            use_reranking=True,
            top_k_search=5,
            top_k_final=3
        )
        
        print("\n  WITHOUT Reranking:")
        for i, doc in enumerate(without["results"], 1):
            score = doc.get('similarity_score', 0)
            print(f"    {i}. [{score:.4f}] {doc['text'][:65]}...")
        
        print("\n  WITH Reranking:")
        for i, doc in enumerate(with_rerank["results"], 1):
            score = doc.get('rerank_score', doc.get('similarity_score', 0))
            print(f"    {i}. [{score:.4f}] {doc['text'][:65]}...")
        
        evaluation_results.append({
            "query": query,
            "without_reranking": without["results"],
            "with_reranking": with_rerank["results"]
        })
    
    print("\n" + "="*70)
    print("✅ Evaluation Complete!")
    print("="*70)
else:
    print("⚠️  Complete previous steps first")


RAG SYSTEM EVALUATION: Testing Multiple Queries

📌 Query: What are the penalties for AI violations?
----------------------------------------------------------------------

  WITHOUT Reranking:
    1. [0.6313] Prohibited AI practices...
    2. [0.5276] Obligations of pro viders of high-r isk AI systems...
    3. [0.5216] Procedure at national lev el f or dealing with AI systems present...

  WITH Reranking:
    1. [0.0367] Prohibited AI practices...
    2. [0.0004] Compliant AI systems which present a r isk...
    3. [0.0004] Classification of AI sy stems as high-r isk...

📌 Query: What is high-risk AI?
----------------------------------------------------------------------

  WITHOUT Reranking:
    1. [0.7735] HIGH-RISK AI SYSTEMS...
    2. [0.7482] Classification of AI sy stems as high-r isk...
    3. [0.6903] Requir ements f or high-r isk AI systems...

  WITH Reranking:
    1. [0.9978] HIGH-RISK AI SYSTEMS...
    2. [0.9783] EU D A T ABASE FOR HIGH-RISK AI SYSTEMS...
    3. [0.0059]

## STEP 10: Create Comparison Table

In [15]:
if 'evaluation_results' in locals():
    # Create a nice comparison table
    
    comparison_data = []
    
    for eval_item in evaluation_results:
        query = eval_item["query"]
        
        # Get top result from each method
        without_top = eval_item["without_reranking"][0] if eval_item["without_reranking"] else None
        with_top = eval_item["with_reranking"][0] if eval_item["with_reranking"] else None
        
        comparison_data.append({
            "Query": query,
            "Without Reranking (Top Result)": without_top["text"][:50] + "..." if without_top else "No result",
            "With Reranking (Top Result)": with_top["text"][:50] + "..." if with_top else "No result",
            "Score Improved": "Yes" if with_top else "No"
        })
    
    # Create DataFrame and display
    df_comparison = pd.DataFrame(comparison_data)
    
    print("\n" + "="*100)
    print("COMPARISON TABLE: Results WITH vs WITHOUT Reranking")
    print("="*100 + "\n")
    print(df_comparison.to_string(index=False))
    
    # Save to CSV for documentation
    df_comparison.to_csv("rag_comparison_results.csv", index=False)
    print("\n✅ Results saved to: rag_comparison_results.csv")
else:
    print("⚠️  Run evaluation first (STEP 9)")


COMPARISON TABLE: Results WITH vs WITHOUT Reranking

                                    Query                        Without Reranking (Top Result)                           With Reranking (Top Result) Score Improved
What are the penalties for AI violations?                            Prohibited AI practices...                            Prohibited AI practices...            Yes
                    What is high-risk AI?                               HIGH-RISK AI SYSTEMS...                               HIGH-RISK AI SYSTEMS...            Yes
 Transparency requirements for AI systems T ransparency obligations f or pro viders and depl... TRANSP ARENCY OBLIGA TIONS F OR PR O VIDERS AND DE...            Yes
                  Prohibited AI practices                            Prohibited AI practices...                            Prohibited AI practices...            Yes
                How should AI be audited? Procedure at national lev el f or dealing with AI ...         Compliant AI syst

## STEP 11: Final Summary

In [16]:
print("\n" + "="*70)
print("✅ LAB COMPLETE!")
print("="*70)

print("""
📊 WHAT YOU'VE CREATED:

  1. ✅ Loaded documents from PDF
  2. ✅ Generated embeddings (OpenAI)
  3. ✅ Uploaded to vector database (Pinecone)
  4. ✅ Built search system
  5. ✅ Implemented reranking (Cohere)
  6. ✅ Compared WITH vs WITHOUT reranking
  7. ✅ Generated results CSV


💡 KEY INSIGHTS:
  • When did reranking improve results most?
  • For which queries did it NOT help as much?
  • What's the tradeoff between quality and cost?

""")

print("="*70)


✅ LAB COMPLETE!

📊 WHAT YOU'VE CREATED:

  1. ✅ Loaded documents from PDF
  2. ✅ Generated embeddings (OpenAI)
  3. ✅ Uploaded to vector database (Pinecone)
  4. ✅ Built search system
  5. ✅ Implemented reranking (Cohere)
  6. ✅ Compared WITH vs WITHOUT reranking
  7. ✅ Generated results CSV


💡 KEY INSIGHTS:
  • When did reranking improve results most?
  • For which queries did it NOT help as much?
  • What's the tradeoff between quality and cost?


